# NanoForecast v0.5 — Colab T4 Training

**v0.5 changes vs v0.3:**
- `USE_DART_NORM = True` → DART-Norm (causal mean/std) instead of median/IQR
- `USE_MULTI_HORIZON = True` → random prediction horizon lengths each batch
- `NanoForecastLoss` → MSE next-token + quantile only (drops anomaly/smoothness)

**To reproduce v0.3 exactly:** set both toggles to `False`.

**Resumable:** If session drops, re-run "Run all" — picks up from last checkpoint.
**Drive usage:** ~50 MB.

---

In [ ]:
# ── GPU check + install ──
import torch, sys, os, json, time, math, random, shutil, warnings, subprocess
warnings.filterwarnings("ignore")
print(f"PyTorch {torch.__version__} | Python {sys.version}")
cuda_ok = torch.cuda.is_available()
if not cuda_ok:
    raise SystemExit("ERROR: No GPU. Runtime → Change runtime type → T4 GPU.")
props = torch.cuda.get_device_properties(0)
vram_gb = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB")

# Install from v0.5 branch
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade",
     "git+https://github.com/eulogik/NanoForecast.git@v0.5",
     "safetensors", "matplotlib"],
    capture_output=True, text=True, env=env,
)
if r.returncode != 0:
    print("pip install from git failed, trying archive download...")
    subprocess.run(
        ["curl", "-sL", "https://github.com/eulogik/NanoForecast/archive/refs/heads/v0.5.tar.gz",
         "-o", "/tmp/nanoforecast.tar.gz"],
        check=True, capture_output=True,
    )
    subprocess.run(["tar", "xzf", "/tmp/nanoforecast.tar.gz", "-C", "/tmp"], check=True)
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--upgrade",
         "/tmp/NanoForecast-0.5", "safetensors", "matplotlib"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit(1)

for mod in list(sys.modules):
    if "nanoforecast" in mod:
        del sys.modules[mod]
import nanoforecast
print("nanoforecast v0.5 installed OK")

# Clear corrupted dataset cache
cache_dir = os.path.expanduser("~/.cache/nanoforecast/datasets")
if os.path.isdir(cache_dir):
    for fname in os.listdir(cache_dir):
        fpath = os.path.join(cache_dir, fname)
        if os.path.isfile(fpath) and os.path.getsize(fpath) == 0:
            os.remove(fpath)
            print(f"  removed empty cache: {fname}")

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/nanoforecast-v05"
os.makedirs(f"{DRIVE_ROOT}/final", exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")

# Imports
import numpy as np
from nanoforecast.config import NanoForecastConfig
from nanoforecast.model.core import NanoForecast
from nanoforecast.data.generator import SyntheticTimeSeriesGenerator
from nanoforecast.data.real_datasets import (
    WindowSpec, build_mixed_pretraining_corpus, time_based_split,
)
from nanoforecast.data.pipeline import create_dataloader
from nanoforecast.train.loss import MultiTaskLoss, NanoForecastLoss
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch import nn

## 1. Configuration

Set the toggles below to choose your experiment:
- **v0.3 reproduction:** `USE_DART_NORM=False`, `USE_MULTI_HORIZON=False`
- **v0.5 DART-Norm only:** `USE_DART_NORM=True`, `USE_MULTI_HORIZON=False`
- **v0.5 full recipe:** `USE_DART_NORM=True`, `USE_MULTI_HORIZON=True`

In [ ]:
# ── Ablation toggles ──
USE_DART_NORM = False       # ← True for DART-Norm (causal mean/std)
USE_MULTI_HORIZON = False   # ← True for random horizon lengths

# ── Model config (matches v0.3) ──
CONFIG = NanoForecastConfig(
    context_length=512,
    prediction_length=48,
    d_model=96,
    num_layers=8,
    patch_size=8,
    covariate_dim=4,
    use_dart_norm=USE_DART_NORM,
)

# ── Training hyperparams (matches v0.3) ──
TRAIN = dict(
    epochs=200,
    batch_size=128,
    lr=3e-5,
    weight_decay=0.01,
    clip_grad=1.0,
    val_fraction=0.2,
    stride=16,
    max_channels=4,
    synthetic_records=10_000,
    datasets=["ETTh1", "ETTh2", "ETTm1", "exchange_rate", "electricity", "traffic"],
    seed=42,
    checkpoint_interval=5,
)

# ── Loss ──
if USE_DART_NORM or USE_MULTI_HORIZON:
    LOSS_FN = NanoForecastLoss(quantiles=CONFIG.quantiles)
    LOSS_NAME = "NanoForecastLoss (next-token + quantile)"
else:
    LOSS_FN = MultiTaskLoss(
        quantiles=CONFIG.quantiles,
        w_point=0.5, w_quantile=1.0, w_anomaly=0.1, w_smooth=0.05,
    )
    LOSS_NAME = "MultiTaskLoss (point + quantile + anomaly + smooth)"

# ── Paths ──
STATE_PATH = f"{DRIVE_ROOT}/training_state.pt"
FINAL_DIR = f"{DRIVE_ROOT}/final"

print(f"DART-Norm: {USE_DART_NORM} | Multi-horizon: {USE_MULTI_HORIZON}")
print(f"Loss: {LOSS_NAME}")
print(f"Model: d_model={CONFIG.d_model}, layers={CONFIG.num_layers}, ctx={CONFIG.context_length}")
print(f"Datasets: {TRAIN['datasets']}")
print(f"Synthetic records: {TRAIN['synthetic_records']:,}")

In [ ]:
# ── Resume detection ──
def _final_exists():
    return os.path.isfile(f"{FINAL_DIR}/config.json") and (
        os.path.isfile(f"{FINAL_DIR}/model.safetensors") or os.path.isfile(f"{FINAL_DIR}/model.pt")
    )

def detect_resume():
    if _final_exists():
        print("Final model found → training already complete. Skipping to benchmark.")
        return {"action": "skip", "epoch": 0, "best_val": float("inf")}
    if os.path.isfile(STATE_PATH):
        state = torch.load(STATE_PATH, map_location="cpu", weights_only=False)
        ep = state.get("epoch", 0)
        best = state.get("best_val_loss", float("inf"))
        print(f"Found training state @ epoch {ep} (best_val={best:.4f}) → resuming.")
        return {"action": "resume", "epoch": ep, "best_val": best, "state": state}
    print("No existing state → starting fresh.")
    return {"action": "fresh", "epoch": 0, "best_val": float("inf")}

RESUME = detect_resume()

## 2. Data Loading

In [ ]:
seed = TRAIN["seed"]
random.seed(seed)
np.random.seed(seed)

spec = WindowSpec(
    context_len=CONFIG.context_length,
    prediction_len=CONFIG.prediction_length,
    stride=TRAIN["stride"],
)

print("Loading real datasets...")
real = build_mixed_pretraining_corpus(
    spec, datasets=TRAIN["datasets"], max_channels_per_dataset=TRAIN["max_channels"]
)
print(f"  real: {len(real)} windows")

print("Generating synthetic data...")
gen = SyntheticTimeSeriesGenerator(seed=seed)
syn = gen.generate_dataset(
    num_series=TRAIN["synthetic_records"],
    context_len=CONFIG.context_length,
    prediction_len=CONFIG.prediction_length,
)
print(f"  synthetic: {len(syn)} windows")

all_records = real + syn
train_records, val_records = time_based_split(all_records, val_fraction=TRAIN["val_fraction"])
print(f"  train: {len(train_records)} | val: {len(val_records)}")

train_loader = create_dataloader(
    train_records, batch_size=TRAIN["batch_size"],
    augment=True, shuffle=True, drop_last=False,
    multi_horizon=USE_MULTI_HORIZON,
)
val_loader = create_dataloader(
    val_records, batch_size=TRAIN["batch_size"],
    augment=False, shuffle=False, drop_last=False,
    multi_horizon=USE_MULTI_HORIZON,
)
print(f"  train batches: {len(train_loader)} | val batches: {len(val_loader)}")

## 3. Model

In [ ]:
model = NanoForecast(CONFIG).cuda()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_params:,} ({n_params/1e6:.2f}M)")

loss_fn = LOSS_FN
optimizer = AdamW(model.parameters(), lr=TRAIN["lr"], weight_decay=TRAIN["weight_decay"])

steps_per_epoch = len(train_loader)
scheduler = OneCycleLR(
    optimizer,
    max_lr=TRAIN["lr"] * 10,
    epochs=TRAIN["epochs"],
    steps_per_epoch=steps_per_epoch,
    pct_start=0.1,
    anneal_strategy="cos",
)

START_EPOCH = 0
best_val_loss = float("inf")
best_epoch = 0
history = []

if RESUME["action"] == "resume":
    s = RESUME["state"]
    model.load_state_dict(s["model_state_dict"])
    optimizer.load_state_dict(s["optimizer_state_dict"])
    START_EPOCH = s["epoch"]
    best_val_loss = s["best_val_loss"]
    best_epoch = s.get("best_epoch", 0)
    history = s.get("history", [])
    n_done = START_EPOCH * steps_per_epoch
    for _ in range(n_done):
        scheduler.step()
    print(f"Resumed at epoch {START_EPOCH}")
elif RESUME["action"] == "skip":
    START_EPOCH = TRAIN["epochs"]

print(f"Training: epochs {START_EPOCH + 1} → {TRAIN['epochs']}")

## 4. Training

In [ ]:
# ── Training loop (resumable, self-checkpointing) ──
if START_EPOCH >= TRAIN["epochs"]:
    print("Training already complete. Skipping.")
else:
    for epoch in range(START_EPOCH + 1, TRAIN["epochs"] + 1):
        t0 = time.time()

        # ── Train ──
        model.train()
        train_losses = {}
        for batch in train_loader:
            x = batch["x"].cuda()
            y = batch["y"].cuda()
            fid = batch["freq_id"].cuda()
            cov = batch["covariates"].cuda() if "covariates" in batch else None
            horizons = batch.get("horizon")

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(x, fid, cov)
                if horizons is not None:
                    # Multi-horizon: per-sample loss
                    B = y.shape[0]
                    losses = []
                    for i in range(B):
                        h = int(horizons[i].item())
                        out_i = {}
                        for k, v in outputs.items():
                            if k in ("loc", "scale"):
                                out_i[k] = v[i:i+1]
                            elif v.dim() >= 3 and v.shape[-1] == CONFIG.prediction_length:
                                out_i[k] = v[i:i+1, ..., :h]
                            elif v.dim() >= 3:
                                out_i[k] = v[i:i+1]
                            else:
                                out_i[k] = v[i:i+1]
                        loss_i, _ = loss_fn(out_i, y[i:i+1, ..., :h], x[i:i+1])
                        losses.append(loss_i)
                    loss = torch.stack(losses).mean()
                    ld = {"loss_total": loss.item()}
                else:
                    loss, ld = loss_fn(outputs, y, x)

            loss.backward()
            if TRAIN["clip_grad"] > 0:
                nn.utils.clip_grad_norm_(model.parameters(), TRAIN["clip_grad"])
            optimizer.step()
            scheduler.step()

            for k, v in ld.items():
                train_losses[k] = train_losses.get(k, 0.0) + v

        for k in train_losses:
            train_losses[k] /= len(train_loader)

        # ── Validate ──
        model.eval()
        val_losses = {}
        with torch.no_grad():
            for batch in val_loader:
                x = batch["x"].cuda()
                y = batch["y"].cuda()
                fid = batch["freq_id"].cuda()
                cov = batch["covariates"].cuda() if "covariates" in batch else None
                horizons = batch.get("horizon")
                with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                    outputs = model(x, fid, cov)
                    if horizons is not None:
                        B = y.shape[0]
                        losses = []
                        for i in range(B):
                            h = int(horizons[i].item())
                            out_i = {}
                            for k, v in outputs.items():
                                if k in ("loc", "scale"):
                                    out_i[k] = v[i:i+1]
                                elif v.dim() >= 3 and v.shape[-1] == CONFIG.prediction_length:
                                    out_i[k] = v[i:i+1, ..., :h]
                                elif v.dim() >= 3:
                                    out_i[k] = v[i:i+1]
                                else:
                                    out_i[k] = v[i:i+1]
                            loss_i, _ = loss_fn(out_i, y[i:i+1, ..., :h], x[i:i+1])
                            losses.append(loss_i)
                        _, ld = torch.stack(losses).mean(), {"loss_total": torch.stack(losses).mean().item()}
                    else:
                        _, ld = loss_fn(outputs, y, x)
                for k, v in ld.items():
                    val_losses[f"val_{k}"] = val_losses.get(f"val_{k}", 0.0) + v

        for k in val_losses:
            val_losses[k] /= len(val_loader)

        elapsed = time.time() - t0
        val_total = val_losses.get("val_loss_total", 0.0)
        train_total = train_losses.get("loss_total", 0.0)
        print(f"E {epoch:03d}/{TRAIN['epochs']:03d} | "
              f"loss={train_total:.4f} | val_loss={val_total:.4f} | "
              f"{elapsed:.0f}s | lr={scheduler.get_last_lr()[0]:.2e}")

        metrics = {"epoch": epoch, **train_losses, **val_losses, "time_s": elapsed}
        history.append(metrics)

        is_best = val_total < best_val_loss
        if is_best:
            best_val_loss = val_total
            best_epoch = epoch

        if epoch % TRAIN["checkpoint_interval"] == 0 or is_best or epoch == TRAIN["epochs"]:
            ckpt = {
                "epoch": epoch,
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": CONFIG,
                "history": history,
                "train_kw": TRAIN,
            }
            torch.save(ckpt, STATE_PATH)
            if is_best:
                print(f"  New best val_loss={best_val_loss:.4f} (saved)")

        if epoch % 20 == 0:
            alloc = torch.cuda.memory_allocated() / 1e9
            res = torch.cuda.memory_reserved() / 1e9
            print(f"  VRAM: {alloc:.1f}GB alloc / {res:.1f}GB reserved")

    # ── Save final artifact ──
    model.eval()
    model.save_pretrained(FINAL_DIR)
    meta = {
        "model_name": "NanoForecast",
        "profile": f"d{CONFIG.d_model}-L{CONFIG.num_layers}{'-dart' if USE_DART_NORM else ''}",
        "params": n_params,
        "config": {
            "context_length": CONFIG.context_length,
            "prediction_length": CONFIG.prediction_length,
            "d_model": CONFIG.d_model,
            "num_layers": CONFIG.num_layers,
            "use_dart_norm": USE_DART_NORM,
            "multi_horizon": USE_MULTI_HORIZON,
        },
        "training": {
            "datasets": TRAIN["datasets"],
            "synthetic_records": TRAIN["synthetic_records"],
            "epochs": TRAIN["epochs"],
            "lr": TRAIN["lr"],
            "batch_size": TRAIN["batch_size"],
            "best_epoch": best_epoch,
            "best_val_loss": best_val_loss,
            "wall_time_s": sum(m.get("time_s", 0) for m in history),
            "loss": LOSS_NAME,
            "dart_norm": USE_DART_NORM,
            "multi_horizon": USE_MULTI_HORIZON,
        },
    }
    with open(f"{FINAL_DIR}/model_card.json", "w") as fh:
        json.dump(meta, fh, indent=2, default=str)
    print(f"Training complete! Best epoch={best_epoch} (val_loss={best_val_loss:.4f})")
    print(f"Final artifact: {FINAL_DIR}")

## 5. Cleanup + Verify

In [ ]:
if os.path.isfile(STATE_PATH):
    os.remove(STATE_PATH)
    print("Removed training_state.pt")

for fn in ["config.json", "model_card.json", "model.safetensors", "model.pt"]:
    path = f"{FINAL_DIR}/{fn}"
    if os.path.isfile(path):
        print(f"  {fn}: {os.path.getsize(path)/1e3:.0f} KB")

## 6. Benchmark

In [ ]:
SKIP_BENCH = False

if not SKIP_BENCH and os.path.isfile(f"{FINAL_DIR}/config.json"):
    tag = "v05"
    if USE_DART_NORM:
        tag += "-dart"
    if USE_MULTI_HORIZON:
        tag += "-mh"
    out_path = f"{DRIVE_ROOT}/benchmark-{tag}.json"
    datasets = ",".join(TRAIN["datasets"])

    repo_tmp = "/tmp/nanoforecast-repo"
    if not os.path.isdir(repo_tmp):
        subprocess.run(
            ["git", "clone", "--depth=1", "--branch", "v0.5",
             "https://github.com/eulogik/NanoForecast.git", repo_tmp],
            check=True, capture_output=True,
        )

    print(f"Benchmarking on {len(TRAIN['datasets'])} datasets...")
    subprocess.run(
        [sys.executable, f"{repo_tmp}/benchmark.py",
         "--checkpoint", FINAL_DIR,
         "--datasets", datasets,
         "--max-windows", "128",
         "--output", out_path,
         "--device", "cuda"],
        check=True,
    )

    if os.path.isfile(out_path):
        with open(out_path) as f:
            results = json.load(f)
        print(f"\n{'='*60}")
        print(f"{'Dataset':<20} {'MASE':<10} {'sMAPE':<10} {'MAE':<10} {'CRPS':<10}")
        print(f"{'-'*60}")
        for name, m in results.get("datasets", {}).items():
            if "error" in m:
                print(f"{name:<20} ERROR: {m['error']}")
            else:
                print(f"{name:<20} {m['mase']:<10.3f} {m['smape']:<10.2f} {m['mae']:<10.3f} {m['crps']:<10.3f}")
        if "overall" in results:
            o = results["overall"]
            print(f"{'-'*60}")
            print(f"{'OVERALL':<20} {o['mase']:<10.3f} {o['smape']:<10.2f} {o['mae']:<10.3f} {o['crps']:<10.3f}")
else:
    print("Benchmark skipped.")

## 7. Training Curves

In [ ]:
if history:
    import matplotlib.pyplot as plt
    epochs_ = [m["epoch"] for m in history]
    train_losses_ = [m.get("loss_total", 0) for m in history]
    val_losses_ = [m.get("val_loss_total", 0) for m in history]

    plt.figure(figsize=(10, 5))
    plt.plot(epochs_, train_losses_, label="Train loss")
    plt.plot(epochs_, val_losses_, label="Val loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    title = f"NanoForecast v0.5"
    if USE_DART_NORM:
        title += " + DART-Norm"
    if USE_MULTI_HORIZON:
        title += " + Multi-Horizon"
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{DRIVE_ROOT}/training_curve.png", dpi=150)
    plt.show()

## 8. Push to Hugging Face (Optional)

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata

try:
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if not token:
    print("No HF_TOKEN. Set it in Colab Secrets (key icon) then re-run.")

if token and os.path.isfile(f"{FINAL_DIR}/config.json"):
    tag = "v05"
    if USE_DART_NORM:
        tag += "-dart"
    if USE_MULTI_HORIZON:
        tag += "-mh"
    REPO_ID = f"eulogik/nanoforecast-{tag}"

    api = HfApi(token=token)
    api.create_repo(repo_id=REPO_ID, private=False, exist_ok=True)
    api.upload_folder(
        folder_path=FINAL_DIR,
        repo_id=REPO_ID,
        commit_message=f"Upload NanoForecast {tag} checkpoint",
    )
    print(f"Pushed to https://huggingface.co/{REPO_ID}")

    bm_path = f"{DRIVE_ROOT}/benchmark-{tag}.json"
    if os.path.isfile(bm_path):
        api.upload_file(path_or_fileobj=bm_path, path_in_repo=f"benchmark-{tag}.json", repo_id=REPO_ID)
else:
    print("Skipping push.")